# Introduction

In this notebook we will:
- Look at cuurent v1 of `pytorch-forecasting`
    - Issues with v1
    - A basic eg of v1 flow
- Introduction to v2
    - The layered approach (to solve the issues with v1)
        - explaination of each layer
    - A basic eg to explain how v2 makes the flow easier and more user-friendly

`pytorch-forecasting` is a package for time series forecasting using deep learning mdoels - built on `torch` and using `lightning` for training abstractions.

### `pytorch-forecasting` v1

- Provides some SoTA models like `TemporalFusionTransformer`, `NBeats`, `NHiTS` etc.
- `lightning` makes the training and testing of the models quite easy 
    - use `trainer` rather than writing training loop yourself
    - provides a wide range of callbacks for functionalities like early stopping, logging etc.

#### Issues with v1

- Highly coupled modules
    - to initialise a model, you have to use `from_dataset` 
    - the boundary between Dataset layer and model layer is not clearly defined
    - overcomplicated boiler plate for implementing the models
    - Doesnot fully utilize the `lightning` functionalities 
        - like there is no `LightningDataModule` that can be used to abstract the dataloader and preprocessing
- A single class - `TimeSeriesDataset` is doing everything 
    -  from preprocessing to dataloading (and somehow even model initialisation!)
        - making it very complicated and hard to understand completely
    - `LightningDataModule` could've made this implementation simpler and the complete flow more easier for the user
        - Just create a model and datamodule and pass to `Trainer` and rest happens behind the curtains!
- No clear `fit` and `predict` contract between models - need to use `lightning.trainer` instead

Lets try out v1 to see what I mean!

In [44]:
import warnings
warnings.filterwarnings("ignore")

In [45]:
import lightning.pytorch as L
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

### Create Synthetic data
We generate a synthetic dataset using `load_toydata` that creates a `pandas` DataFrame with just numerical values as for now **the pipeline assumes the data to be numerical only**.

In [46]:
from utils import load_toydata

In [47]:
num_series = 100  # Number of individual time series to generate
seq_length = 50  # Length of each time series
df = load_toydata(num_series, seq_length)
df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,0.101733,0.136985,0,1.000000,0.581424,0
1,0,1,0.136985,0.338498,0,0.995004,0.581424,0
2,0,2,0.338498,0.640482,0,0.980067,0.581424,0
3,0,3,0.640482,0.742722,0,0.955336,0.581424,0
4,0,4,0.742722,0.869786,0,0.921061,0.581424,0


In [48]:
# v1 needs categoricals as strings
for col in ["category", "static_feature_cat"]:
    df[col] = df[col].astype(str)

In [49]:
max_encoder_length = 30
max_prediction_length = 6
cutoff = df["time_idx"].max() - max_prediction_length

Create the dataset (which does everything - windowing, scaling etc)

In [50]:
training_dataset = TimeSeriesDataSet(
    df[df.time_idx <= cutoff],
    time_idx="time_idx",
    target="y",
    group_ids=["series_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_reals=["static_feature"],
    static_categoricals=["static_feature_cat"],
    time_varying_known_reals=["time_idx", "future_known_feature"],
    time_varying_unknown_reals=["y", "x"],
    time_varying_unknown_categoricals=["category"],
    target_normalizer=GroupNormalizer(groups=["series_id"]),
)
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, df, predict=True, stop_randomization=True
)



Create the data loaders (which are created by the dataset class itself)


In [51]:
train_dataloader = training_dataset.to_dataloader(train=True, batch_size=64, num_workers=0)
val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=64, num_workers=0)


Now you cant initialise the model using `__init__`, but by using `from_dataset` as the info about the dataset is taken from the dataset class directly using this

In [52]:
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=2,
    dropout=0.1,
    loss=QuantileLoss(),
)


If you dont want to use `from_dataset`, then you have to pass everything yourself
```python
tft = TemporalFusionTransformer(
    # architecture
    hidden_size=32,
    lstm_layers=1,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,
    learning_rate=1e-3,
    # loss + output width must agree
    loss=loss,
    output_size=len(loss.quantiles),
    # windowing
    max_encoder_length=30,
    # the metadata from_dataset would have derived
    static_categoricals=static_categoricals,
    static_reals=static_reals,
    time_varying_categoricals_encoder=time_varying_categoricals_encoder,
    time_varying_categoricals_decoder=time_varying_categoricals_decoder,
    time_varying_reals_encoder=time_varying_reals_encoder,
    time_varying_reals_decoder=time_varying_reals_decoder,
    x_reals=x_reals,
    x_categoricals=x_categoricals,
    embedding_sizes=embedding_sizes,
    embedding_paddings=[],
    categorical_groups={},
    # needed for predict() to return original scale
    output_transformer=training.target_normalizer,
)
```

Use the trainer to `fit` and `predict`

In [53]:
# train
trainer = L.Trainer(max_epochs=5, accelerator="auto", gradient_clip_val=0.1)
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

# predict
preds = tft.predict(val_dataloader, return_index=True, return_x=True)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   

Epoch 4: 100%|██████████| 12/12 [00:00<00:00, 18.17it/s, v_num=11, train_loss_step=1.100, val_loss=24.70, train_loss_epoch=1.780]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 12/12 [00:00<00:00, 17.08it/s, v_num=11, train_loss_step=1.100, val_loss=24.70, train_loss_epoch=1.780]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


## `pytorch-forecasting` v2

`pytorch-forecasting` v2 solves these issues of v1 using a layered approach

4 layers:
- two for data (D1, D2)
- one for models (M)
- one layer as a abstraction "package" layer for the data layers and model layer (P).

```
DataFrame → [D1 TimeSeries] → [D2 DataModule] → [M Model]
                             └──   Package (P) Layer   ──┘       
```

The metadata that the v1 passes to the models using `from_dataset` is now sent using a basic `metadata` dict, and this dict is created by `D2` layer and is sent to the `M` layer. But these layers never are called INSIDE the classes like they were called in the v1


**Core idea:** models never touch pandas; data objects never know which model will use them. They meet through `metadata`.

## D1 — `TimeSeries`
 
Holds your raw data as tensors, and tags what each column *is*.
 
- Three independent axes:
  - **type** → `num` or `cat`
  - **time behaviour** → `static` or not
  - **future availability** → `known` or `unknown`
- A column can appear on more than one axis (e.g. `num` *and* `unknown`).
- Also needs: `time`, `target`, `group` (what identifies one series).

Does **not**: window, scale, split, or build dataloaders.

---

## D2 — the DataModule
 
A `LightningDataModule`. Turns D1 tensors into training batches.
 
- **Windowing** — `max_encoder_length`, `max_prediction_length`
- **Preprocessing** — `scalers`, `categorical_encoders`, `target_normalizer`
- **Splitting** — `train_val_test_split`, default `(0.7, 0.15, 0.15)`
- **Batching** — `train/val/test_dataloader()`

We can have multiple datamodules (for different families of models).

Like currently we have two families of models in v2, so we have two types of datamodules

- `EncoderDecoderTimeSeriesDataModule` — encoder–decoder models (TFT)
- `TslibDataModule` — tslib models (TimeXer, Informer, AutoFormer)

**`metadata`** — D2's description of the batches it produces (lengths, feature
counts). The model reads this to size itself.
 
---

## M — the model
 
A plain `LightningModule`. The network plus its training and prediction steps.
 
- Data-agnostic: no pandas, no column names
- Gets `metadata=dm.metadata` instead of v1's `from_dataset()`

Use directly when you want your own Trainer, callbacks, or training loop.
 
---
 
## P — the package
 
The orchestrator. Takes three config dicts and runs everything.
 
- `datamodule_cfg`, `model_cfg`, `trainer_cfg`
- `sklearn`-style `pkg.fit()` / `pkg.predict()`, plus checkpointing
- Also holds tags registry and test fixtures

**Changed from v1:** the P layer used to be internal, for testing only. In v2 it
is the high-level user-facing API.

Lets try out the same example above using v2 pipeline

We use the same dataset

In [54]:
from pytorch_forecasting.data.timeseries import TimeSeries
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from sklearn.preprocessing import StandardScaler
from pytorch_forecasting.data.encoders import (
    NaNLabelEncoder,
    TorchNormalizer,
)
from pytorch_forecasting.metrics import MAE, SMAPE
from pytorch_forecasting.models.temporal_fusion_transformer._tft_pkg_v2 import (
    TFT_pkg_v2,
)

In [55]:
num_series = 100  # Number of individual time series to generate
seq_length = 50  # Length of each time series
df = load_toydata(num_series, seq_length)
df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,0.062623,0.179141,0,1.000000,0.900066,0
1,0,1,0.179141,0.494013,0,0.995004,0.900066,0
2,0,2,0.494013,0.553875,0,0.980067,0.900066,0
3,0,3,0.553875,0.781289,0,0.955336,0.900066,0
4,0,4,0.781289,0.843028,0,0.921061,0.900066,0


Pass this `df` to D1 (or `TimeSeries`) layer

In [56]:
# create `TimeSeries` dataset that returns the raw data in terms of tensors
dataset = TimeSeries(
    data=df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

Now here the things get more easy using the `pkg` layer.
Just create the configs and rest happens under the hood.

The configs:
- `datamodule_cfg` - for the datamodule (D2)
- `model_cfg` - for the model (M)
- `trainer_cfg` - for the `trainer`

In [57]:
datamodule_cfg = dict(
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

In [58]:
model_cfg = dict(
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    lr_scheduler="reduce_lr_on_plateau",
    lr_scheduler_params={"mode": "min", "factor": 0.1, "patience": 10},
    hidden_size=64,
    num_layers=2,
    attention_head_size=4,
    dropout=0.1,
)

In [59]:
trainer_cfg = dict(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

### Create the `model_pkg` object

This `pkg` class acts as a wrapper around the whole ML pipeline in `pytorch-forecasting` and we can simply just define the `pkg` class and then use `pkg.fit` and `pkg.predict` to perform the "fit", "predict" mechanisms.

In [60]:
model_pkg = TFT_pkg_v2(
    model_cfg=model_cfg,
    trainer_cfg=trainer_cfg,
    datamodule_cfg=datamodule_cfg,
)

{'loss': MAE(), 'logging_metrics': [MAE(), SMAPE()], 'optimizer': 'adam', 'optimizer_params': {'lr': 0.001}, 'lr_scheduler': 'reduce_lr_on_plateau', 'lr_scheduler_params': {'mode': 'min', 'factor': 0.1, 'patience': 10}, 'hidden_size': 64, 'num_layers': 2, 'attention_head_size': 4, 'dropout': 0.1}


In [61]:
model_pkg.fit(dataset)  # You can also pass in a DataModule here

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------------------
0 | loss                  | MAE                | 0      | train | 0    
1 | logging_metrics       | ModuleList         | 0      | train | 0    
2 | encoder_var_selection | Sequential         | 709    | train | 0    
3 | decoder_var_selection | Sequential         | 193    | train | 0    
4 | static_context_linear | Linear             | 192    | train | 0    
5 | lstm_encoder          | LSTM               | 51.5 K | train | 0    
6 | lstm_decoder          | LSTM               | 50.4 K | train |

Epoch 4: 100%|██████████| 42/42 [00:02<00:00, 14.87it/s, v_num=14, train_loss_step=5.720, val_loss=3.080, val_MAE=3.080, val_SMAPE=0.721, train_loss_epoch=4.410, train_MAE=4.410, train_SMAPE=0.657]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 42/42 [00:02<00:00, 14.78it/s, v_num=14, train_loss_step=5.720, val_loss=3.080, val_MAE=3.080, val_SMAPE=0.721, train_loss_epoch=4.410, train_MAE=4.410, train_SMAPE=0.657]
Artifacts saved in: /home/aryan/pytorch-forecasting-v2-user-testing/checkpoints


PosixPath('/home/aryan/pytorch-forecasting-v2-user-testing/checkpoints/best-epoch=4-step=210-v1.ckpt')

In [62]:
preds = model_pkg.predict(dataset, return_info=["index", "x", "y"])
# You can also pass in a DataModule or Dataloader here

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 60/60 [00:02<00:00, 22.09it/s]



## Which layer do I touch?
 
| I want to… | Layer |
|---|---|
| Say what a column means | D1 |
| Change history / horizon length | D2 |
| Scale features, normalize target | D2 |
| Change the split | D2 |
| Support a new batch format | D2 (new DataModule) |
| Change network, loss, optimizer | M |
| Use my own Trainer | M (skip P) |
| Just train and predict | P |
| Save / load a fitted pipeline | P |
| Add a new architecture | M + P |
 

More on the v2 in the next notebook!